# Islamabad House Price Predictor

Predicts property prices in Islamabad using data scraped from Zameen.com. Trains and compares 6 regression models (Linear Regression, Decision Tree, Random Forest, Gradient Boosting, XGBoost, CatBoost), then serves the best model through an interactive Gradio app.

**Pipeline:**
1. Scrape listings from Zameen.com (Selenium + BeautifulSoup)
2. Clean and feature-engineer the raw data (price/area unit conversion, location & property type encoding)
3. Exploratory data analysis
4. Train and compare 6 regression models
5. Launch a Gradio web app for interactive price prediction

> This notebook is designed for Google Colab. Run cells in order. Scraping takes several minutes and is best run once; the cleaned dataset and trained models are saved to Google Drive for reuse.

In [ ]:
# Mount Google Drive (used to persist scraped data and trained models)
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!pip install -q selenium beautifulsoup4 chromedriver-autoinstaller xgboost catboost gradio

In [ ]:
# Imports
import time
import random
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import chromedriver_autoinstaller

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("All libraries loaded successfully!")

## 1. Web scraping (Zameen.com)

Scrapes Islamabad property listings: price, area, location, bedrooms, bathrooms, title, and listing date.

In [ ]:
def create_driver():
    chromedriver_autoinstaller.install()

    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(options=options)

In [ ]:
def scrape_zameen_islamabad(total_pages=20, target_count=400):
    """Scrape Islamabad property listings from Zameen.com.

    Returns a list of dicts with raw fields: price, area, location,
    bedrooms, bathrooms, title, date_added, city, page.
    """
    all_data = []
    driver = create_driver()

    for page_num in range(1, total_pages + 1):
        url = f"https://www.zameen.com/Homes/Islamabad-3-{page_num}.html"
        print(f"Page {page_num}/{total_pages} -> {url}")

        try:
            driver.get(url)
            time.sleep(random.uniform(5, 8))

            # scroll to trigger lazy-loaded listings
            driver.execute_script("window.scrollTo(0, 1500);")
            time.sleep(1.5)
            driver.execute_script("window.scrollTo(0, 3000);")
            time.sleep(1.5)
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)

            soup = BeautifulSoup(driver.page_source, "html.parser")
            listings = soup.find_all("li", attrs={"aria-label": "Listing"})
            print(f"  Found {len(listings)} listings")

            for listing in listings:
                try:
                    currency_tag = listing.find(attrs={"aria-label": "Currency"})
                    currency = currency_tag.get_text(strip=True) if currency_tag else "PKR"

                    price_tag = listing.find(attrs={"aria-label": "Price"})
                    price_raw = price_tag.get_text(strip=True) if price_tag else None
                    price = f"{currency} {price_raw}" if price_raw else None

                    loc_tag = listing.find(attrs={"aria-label": "Location"})
                    location = loc_tag.get_text(strip=True) if loc_tag else None

                    area_tag = listing.find(attrs={"aria-label": "Area"})
                    area = area_tag.get_text(strip=True) if area_tag else None

                    bed_tag = listing.find(attrs={"aria-label": "Beds"})
                    bedrooms = bed_tag.get_text(strip=True) if bed_tag else None

                    bath_tag = listing.find(attrs={"aria-label": "Baths"})
                    bathrooms = bath_tag.get_text(strip=True) if bath_tag else None

                    title_tag = listing.find(attrs={"aria-label": "Title"})
                    title = title_tag.get_text(strip=True) if title_tag else None

                    date_tag = listing.find(attrs={"aria-label": "Listing creation date"})
                    date_added = date_tag.get_text(strip=True) if date_tag else None

                    if price and area and location:
                        all_data.append({
                            "price": price,
                            "area": area,
                            "location": location,
                            "bedrooms": bedrooms,
                            "bathrooms": bathrooms,
                            "title": title,
                            "date_added": date_added,
                            "city": "Islamabad",
                            "page": page_num
                        })

                except Exception:
                    continue

            print(f"  Total collected so far: {len(all_data)}")

            if len(all_data) >= target_count:
                print(f"Reached {target_count} listings!")
                break

            time.sleep(random.uniform(4, 7))

        except Exception as e:
            print(f"Error on page {page_num}: {e}")
            time.sleep(5)
            continue

    driver.quit()
    print(f"Scraping complete! Total: {len(all_data)} listings")
    return all_data

In [ ]:
# Run the scraper (takes several minutes)
data = scrape_zameen_islamabad(total_pages=20, target_count=400)

df_raw = pd.DataFrame(data)
print(f"Shape: {df_raw.shape}")
df_raw.head()

In [ ]:
# Save the raw scraped data
df_raw.to_csv("zameen_raw.csv", index=False)
df_raw.to_csv("/content/drive/MyDrive/zameen_raw.csv", index=False)
print("Saved raw dataset locally and to Google Drive!")

## 2. Data cleaning & feature engineering

Converts price/area strings into numeric values, encodes location and property type, removes outliers and invalid rows.

In [ ]:
def convert_price(price_str):
    """Convert a price string like 'PKR 10.5 Crore' or 'PKR 85 Lakh' to a numeric PKR value."""
    try:
        price_str = str(price_str).replace("PKR", "").replace(",", "").strip()

        if "Crore" in price_str:
            return float(price_str.replace("Crore", "").strip()) * 10_000_000
        elif "Lakh" in price_str:
            return float(price_str.replace("Lakh", "").strip()) * 100_000
        else:
            return float(price_str)
    except Exception:
        return np.nan


def convert_area(area_str):
    """Convert an area string (Kanal, Marla, Sq. Ft., Sq. Yd.) to square feet."""
    try:
        area_str = str(area_str).strip()

        if "Kanal" in area_str:
            return float(area_str.replace("Kanal", "").strip()) * 4500
        elif "Marla" in area_str:
            return float(area_str.replace("Marla", "").strip()) * 225
        elif "Sq. Ft." in area_str or "Sq.Ft." in area_str:
            return float(area_str.replace("Sq. Ft.", "").replace("Sq.Ft.", "").strip())
        elif "Sq. Yd." in area_str:
            return float(area_str.replace("Sq. Yd.", "").strip()) * 9
        else:
            return np.nan
    except Exception:
        return np.nan


def extract_property_type(title):
    """Infer property type (House, Flat, Plot, Farm House, Office) from the listing title."""
    title = str(title).lower()
    if "house" in title:
        return "House"
    elif "flat" in title or "apartment" in title:
        return "Flat"
    elif "plot" in title:
        return "Plot"
    elif "farm" in title:
        return "Farm House"
    elif "office" in title:
        return "Office"
    else:
        return "House"  # default

In [ ]:
df_clean = df_raw.copy()

# Convert price and area to numeric
df_clean["price_pkr"] = df_clean["price"].apply(convert_price)
df_clean["area_sqft"] = df_clean["area"].apply(convert_area)

# Convert bedrooms/bathrooms to numeric, fill missing with median
df_clean["bedrooms"] = pd.to_numeric(df_clean["bedrooms"], errors="coerce")
df_clean["bathrooms"] = pd.to_numeric(df_clean["bathrooms"], errors="coerce")
df_clean["bedrooms"].fillna(df_clean["bedrooms"].median(), inplace=True)
df_clean["bathrooms"].fillna(df_clean["bathrooms"].median(), inplace=True)

# Remove unrealistic bedroom/bathroom counts
df_clean = df_clean[df_clean["bedrooms"].between(1, 15)]
df_clean = df_clean[df_clean["bathrooms"].between(1, 15)]

# Drop rows missing essential fields, and remove duplicates
before = len(df_clean)
df_clean.dropna(subset=["price_pkr", "area_sqft", "location"], inplace=True)
df_clean.drop_duplicates(inplace=True)
print(f"Rows before cleaning: {before} -> after: {len(df_clean)}")

# Remove top/bottom 1% price outliers
low, high = df_clean["price_pkr"].quantile([0.01, 0.99])
df_clean = df_clean[df_clean["price_pkr"].between(low, high)]
print(f"After outlier removal: {len(df_clean)} rows")

In [ ]:
# Clean location (take the first part before the comma) and extract property type
df_clean["location_clean"] = df_clean["location"].apply(lambda x: str(x).split(",")[0].strip())
df_clean["property_type"] = df_clean["title"].apply(extract_property_type)

print("Property types:")
print(df_clean["property_type"].value_counts())
print("\nTop 10 locations:")
print(df_clean["location_clean"].value_counts().head(10))

# Encode categorical features
le_location = LabelEncoder()
le_proptype = LabelEncoder()
df_clean["location_encoded"] = le_location.fit_transform(df_clean["location_clean"])
df_clean["property_type_encoded"] = le_proptype.fit_transform(df_clean["property_type"])

# Save encoders (needed later for the Gradio app)
pickle.dump(le_location, open("le_location.pkl", "wb"))
pickle.dump(le_proptype, open("le_proptype.pkl", "wb"))
pickle.dump(le_location, open("/content/drive/MyDrive/le_location.pkl", "wb"))
pickle.dump(le_proptype, open("/content/drive/MyDrive/le_proptype.pkl", "wb"))

print("\nEncoding done and encoders saved!")

In [ ]:
# Save the cleaned dataset
df_clean.to_csv("zameen_clean.csv", index=False)
df_clean.to_csv("/content/drive/MyDrive/zameen_clean.csv", index=False)
print("Clean dataset saved!")

feature_columns = ["area_sqft", "bedrooms", "bathrooms", "location_encoded", "property_type_encoded"]
X = df_clean[feature_columns]
y = df_clean["price_pkr"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training samples: {len(X_train)}, Testing samples: {len(X_test)}")

## 3. Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(df_clean["price_pkr"] / 1e6, bins=30, color="steelblue", edgecolor="black")
axes[0, 0].set_xlabel("Price (Million PKR)")
axes[0, 0].set_ylabel("Number of Properties")
axes[0, 0].set_title("Price Distribution")

axes[0, 1].hist(df_clean["area_sqft"], bins=30, color="green", edgecolor="black")
axes[0, 1].set_xlabel("Area (Square Feet)")
axes[0, 1].set_ylabel("Number of Properties")
axes[0, 1].set_title("Area Distribution")

bedroom_counts = df_clean["bedrooms"].value_counts().sort_index()
axes[1, 0].bar(bedroom_counts.index, bedroom_counts.values, color="orange", edgecolor="black")
axes[1, 0].set_xlabel("Number of Bedrooms")
axes[1, 0].set_ylabel("Number of Properties")
axes[1, 0].set_title("Bedroom Distribution")

type_counts = df_clean["property_type"].value_counts()
axes[1, 1].bar(type_counts.index, type_counts.values, color="purple", edgecolor="black")
axes[1, 1].set_xlabel("Property Type")
axes[1, 1].set_ylabel("Number of Properties")
axes[1, 1].set_title("Property Type Distribution")

plt.tight_layout()
plt.savefig("preprocessing_graphs.png", dpi=150)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df_clean["area_sqft"], df_clean["price_pkr"] / 1e6, alpha=0.5, color="steelblue")
plt.xlabel("Area (Square Feet)")
plt.ylabel("Price (Million PKR)")
plt.title("Price vs Area - Islamabad Properties")
plt.savefig("price_vs_area.png", dpi=150)
plt.show()

## 4. Model training & comparison

Trains and evaluates 6 regression models, selecting the best by R² score.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=10, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42, verbosity=0),
    "CatBoost": CatBoostRegressor(iterations=100, learning_rate=0.1, depth=6, random_state=42, verbose=0),
}

results = []
trained_models = {}

print(f"{'Model':<25} {'MAE':>15} {'RMSE':>15} {'R\u00b2':>8}")
print("-" * 70)

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    results.append({"Model": name, "MAE": mae, "RMSE": rmse, "R\u00b2": r2})
    trained_models[name] = model

    pickle.dump(model, open(f"{name.replace(' ', '_')}.pkl", "wb"))
    pickle.dump(model, open(f"/content/drive/MyDrive/{name.replace(' ', '_')}.pkl", "wb"))

    print(f"{name:<25} PKR {mae:>12,.0f} PKR {rmse:>12,.0f} {r2:>8.4f}")

print("\nAll 6 models trained and saved!")

In [ ]:
results_df = pd.DataFrame(results).sort_values("R\u00b2", ascending=False).reset_index(drop=True)

print("=" * 70)
print("       FINAL MODEL COMPARISON (sorted by R\u00b2 score)")
print("=" * 70)
print(results_df.to_string(index=False))

best_model_name = results_df.iloc[0]["Model"]
print(f"\nBEST MODEL: {best_model_name}")
print(f"   R\u00b2 Score: {results_df.iloc[0]['R\u00b2']:.4f}")
print(f"   MAE:      PKR {results_df.iloc[0]['MAE']:,.0f}")
print(f"   RMSE:     PKR {results_df.iloc[0]['RMSE']:,.0f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
colors = ["gold" if i == 0 else "steelblue" for i in range(len(results_df))]

axes[0].barh(results_df["Model"], results_df["R\u00b2"], color=colors)
axes[0].set_xlabel("R\u00b2 Score (higher is better)")
axes[0].set_title("R\u00b2 Score Comparison")
for i, v in enumerate(results_df["R\u00b2"]):
    axes[0].text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=9)

axes[1].barh(results_df["Model"], results_df["RMSE"] / 1e6, color="salmon")
axes[1].set_xlabel("RMSE (Million PKR) - lower is better")
axes[1].set_title("RMSE Comparison")
for i, v in enumerate(results_df["RMSE"] / 1e6):
    axes[1].text(v + 0.01, i, f"{v:.2f}M", va="center", fontsize=9)

axes[2].barh(results_df["Model"], results_df["MAE"] / 1e6, color="lightgreen")
axes[2].set_xlabel("MAE (Million PKR) - lower is better")
axes[2].set_title("MAE Comparison")
for i, v in enumerate(results_df["MAE"] / 1e6):
    axes[2].text(v + 0.01, i, f"{v:.2f}M", va="center", fontsize=9)

plt.suptitle("ML Model Performance Comparison - Islamabad House Price Prediction", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Interactive price predictor (Gradio app)

A multi-step web app: welcome screen -> property details form -> price estimate, styled with custom CSS.

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr
import numpy as np
import pickle

# ── Load encoders & best model ───────────────────────────────────────────────
le_loc     = pickle.load(open("le_location.pkl",  "rb"))
le_prop    = pickle.load(open("le_proptype.pkl",   "rb"))
best_model = trained_models[best_model_name]

available_locations = sorted(list(le_loc.classes_))

# ── Prediction logic ──────────────────────────────────────────────────────────
def predict_price(area_value, area_unit, bedrooms, bathrooms, location, property_type):
    try:
        if area_unit == "Marla":
            area_sqft = float(area_value) * 225
        elif area_unit == "Kanal":
            area_sqft = float(area_value) * 4500
        else:
            area_sqft = float(area_value)

        location_encoded = (
            le_loc.transform([location])[0] if location in le_loc.classes_ else 0
        )
        prop_encoded = (
            le_prop.transform([property_type])[0]
            if property_type in le_prop.classes_ else 0
        )

        features = np.array([[
            area_sqft, float(bedrooms), float(bathrooms),
            location_encoded, prop_encoded
        ]])
        price = best_model.predict(features)[0]

        if price >= 1e7:
            price_str = f"PKR {price/1e7:.2f} Crore"
        else:
            price_str = f"PKR {price/1e5:.2f} Lakh"

        # pick a relevant Unsplash photo for the property type
        photo_map = {
            "House":      "https://images.unsplash.com/photo-1564013799919-ab600027ffc6?w=600&q=80",
            "Flat":       "https://images.unsplash.com/photo-1545324418-cc1a3fa10c00?w=600&q=80",
            "Farm House": "https://images.unsplash.com/photo-1568605114967-8130f3a36994?w=600&q=80",
            "Plot":       "https://images.unsplash.com/photo-1500382017468-9049fed747ef?w=600&q=80",
        }
        photo_url = photo_map.get(property_type, photo_map["House"])

        house_svg = _house_svg(property_type, bedrooms)

        result_html = f"""
        <div style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif">

          <!-- price hero banner -->
          <div style="background:linear-gradient(135deg,#0F6E56 0%,#1D9E75 60%,#2EC98E 100%);
                      border-radius:20px;padding:36px 32px;margin-bottom:20px;
                      position:relative;overflow:hidden">
            <!-- decorative circles -->
            <div style="position:absolute;top:-40px;right:-40px;width:180px;height:180px;
                        border-radius:50%;background:rgba(255,255,255,.07)"></div>
            <div style="position:absolute;bottom:-30px;right:80px;width:120px;height:120px;
                        border-radius:50%;background:rgba(255,255,255,.05)"></div>
            <p style="font-size:11px;color:rgba(255,255,255,.7);text-transform:uppercase;
                      letter-spacing:1.2px;margin-bottom:10px">Estimated market value</p>
            <p style="font-size:44px;font-weight:600;color:white;margin:0;line-height:1">
              {price_str}
            </p>
            <p style="font-size:14px;color:rgba(255,255,255,.7);margin-top:8px">
              PKR {price:,.0f} &nbsp;·&nbsp; {location}
            </p>
            <div style="display:inline-flex;align-items:center;gap:6px;background:rgba(255,255,255,.15);
                        border-radius:20px;padding:5px 14px;margin-top:14px">
              <div style="width:6px;height:6px;border-radius:50%;background:#7FFFD4"></div>
              <span style="font-size:12px;color:white">Predicted by {best_model_name}</span>
            </div>
          </div>

          <!-- two column layout -->
          <div style="display:grid;grid-template-columns:1fr 1fr;gap:16px;margin-bottom:16px">

            <!-- property photo -->
            <div style="border-radius:16px;overflow:hidden;height:240px;position:relative">
              <img src="{photo_url}" alt="{property_type}"
                   style="width:100%;height:100%;object-fit:cover"/>
              <div style="position:absolute;bottom:0;left:0;right:0;
                          background:linear-gradient(to top,rgba(0,0,0,.6),transparent);
                          padding:16px 16px 12px">
                <span style="font-size:13px;font-weight:500;color:white">
                  {property_type} · {int(bedrooms)} bed · {int(bathrooms)} bath
                </span>
              </div>
              <div style="position:absolute;top:12px;left:12px;background:rgba(0,0,0,.45);
                          border-radius:20px;padding:4px 12px">
                <span style="font-size:11px;color:white">📍 {location.split(",")[0]}</span>
              </div>
            </div>

            <!-- property summary -->
            <div style="background:white;border:0.5px solid #eee;border-radius:16px;padding:20px">
              <p style="font-size:11px;color:#aaa;text-transform:uppercase;
                        letter-spacing:.8px;margin-bottom:14px">Property details</p>
              {"".join(
                  f"<div style='display:flex;justify-content:space-between;"
                  f"align-items:center;padding:10px 0;"
                  f"border-bottom:0.5px solid #f5f5f5;font-size:13px'>"
                  f"<span style='color:#aaa;display:flex;align-items:center;gap:6px'>"
                  f"{icon} {k}</span>"
                  f"<span style='font-weight:500;color:#1a1a1a'>{v}</span></div>"
                  for icon, k, v in [
                      ("📐", "Area",          f"{area_value} {area_unit} ({area_sqft:,.0f} sq ft)"),
                      ("🛏️", "Bedrooms",      int(bedrooms)),
                      ("🚿", "Bathrooms",     int(bathrooms)),
                      ("🏡", "Type",          property_type),
                      ("📅", "Estimated on",  "Today"),
                  ]
              )}
            </div>
          </div>

          <!-- 3 insight cards -->
          <div style="display:grid;grid-template-columns:repeat(3,1fr);gap:12px;margin-bottom:16px">
            <div style="background:#F0FDF7;border:0.5px solid #C6F0DF;
                        border-radius:14px;padding:18px">
              <p style="font-size:11px;color:#3B6D11;text-transform:uppercase;
                        letter-spacing:.6px;margin-bottom:6px">Per sq ft</p>
              <p style="font-size:22px;font-weight:600;color:#0F6E56;margin:0">
                PKR {price/area_sqft:,.0f}
              </p>
              <p style="font-size:11px;color:#3B6D11;margin-top:4px">rate per sq ft</p>
            </div>
            <div style="background:#FFF8F0;border:0.5px solid #FFE4C0;
                        border-radius:14px;padding:18px">
              <p style="font-size:11px;color:#8B5E3C;text-transform:uppercase;
                        letter-spacing:.6px;margin-bottom:6px">Area size</p>
              <p style="font-size:22px;font-weight:600;color:#7A4520;margin:0">
                {area_sqft:,.0f}
              </p>
              <p style="font-size:11px;color:#8B5E3C;margin-top:4px">square feet</p>
            </div>
            <div style="background:#F3F0FF;border:0.5px solid #D9D0FF;
                        border-radius:14px;padding:18px">
              <p style="font-size:11px;color:#5B4B8A;text-transform:uppercase;
                        letter-spacing:.6px;margin-bottom:6px">Rooms</p>
              <p style="font-size:22px;font-weight:600;color:#3D2E7A;margin:0">
                {int(bedrooms) + int(bathrooms)}
              </p>
              <p style="font-size:11px;color:#5B4B8A;margin-top:4px">total rooms</p>
            </div>
          </div>

          <!-- disclaimer -->
          <div style="background:#f9f9f7;border-radius:12px;padding:14px 18px;
                      display:flex;align-items:center;gap:10px">
            <span style="font-size:18px">💡</span>
            <p style="font-size:12px;color:#999;margin:0;line-height:1.6">
              This estimate is based on <strong style="color:#666">400+ real Islamabad listings</strong>
              from Zameen.com and 6 ML models. Actual prices may vary based on condition,
              negotiation, and market timing.
            </p>
          </div>

        </div>
        """
        return result_html

    except Exception as e:
        return f"<p style='color:#E24B4A;padding:1rem'>Error: {str(e)}</p>"


def _house_svg(prop_type, beds):
    if prop_type == "Flat":
        return """
        <svg width="200" height="160" viewBox="0 0 200 160" xmlns="http://www.w3.org/2000/svg">
          <rect x="20" y="20" width="160" height="130" rx="4" fill="#E1F5EE" stroke="#1D9E75" stroke-width="1.5"/>
          <rect x="35" y="30" width="50" height="28" rx="3" fill="#9FE1CB" stroke="#1D9E75" stroke-width="1"/>
          <rect x="115" y="30" width="50" height="28" rx="3" fill="#9FE1CB" stroke="#1D9E75" stroke-width="1"/>
          <rect x="35" y="70" width="50" height="28" rx="3" fill="#9FE1CB" stroke="#1D9E75" stroke-width="1"/>
          <rect x="115" y="70" width="50" height="28" rx="3" fill="#085041" stroke="#1D9E75" stroke-width="1"/>
          <rect x="35" y="110" width="50" height="28" rx="3" fill="#9FE1CB" stroke="#1D9E75" stroke-width="1"/>
          <rect x="115" y="110" width="50" height="28" rx="3" fill="#9FE1CB" stroke="#1D9E75" stroke-width="1"/>
        </svg>"""
    if prop_type == "Farm House":
        return """
        <svg width="220" height="175" viewBox="0 0 220 175" xmlns="http://www.w3.org/2000/svg">
          <rect x="30" y="90" width="160" height="75" rx="4" fill="#EAF3DE" stroke="#1D9E75" stroke-width="1.5"/>
          <polygon points="110,18 20,90 200,90" fill="#1D9E75" opacity=".9"/>
          <rect x="45" y="108" width="40" height="28" rx="3" fill="#9FE1CB" stroke="#1D9E75" stroke-width="1"/>
          <rect x="135" y="108" width="40" height="28" rx="3" fill="#9FE1CB" stroke="#1D9E75" stroke-width="1"/>
          <rect x="88" y="118" width="44" height="47" rx="3" fill="#085041"/>
          <circle cx="42" cy="160" r="12" fill="#C0DD97" opacity=".8"/>
          <circle cx="55" cy="150" r="15" fill="#9FE1CB" opacity=".8"/>
          <circle cx="172" cy="158" r="11" fill="#C0DD97" opacity=".8"/>
          <circle cx="183" cy="148" r="14" fill="#9FE1CB" opacity=".8"/>
        </svg>"""
    if prop_type == "Plot":
        return """
        <svg width="200" height="150" viewBox="0 0 200 150" xmlns="http://www.w3.org/2000/svg">
          <rect x="20" y="40" width="160" height="90" rx="4" fill="#EAF3DE" stroke="#1D9E75"
                stroke-width="1.5" stroke-dasharray="6,4"/>
          <text x="100" y="92" text-anchor="middle" font-size="13" font-weight="500" fill="#0F6E56">Plot</text>
          <text x="100" y="110" text-anchor="middle" font-size="10" fill="#3B6D11">Vacant land</text>
        </svg>"""
    floors   = 2 if beds >= 5 else 1
    wall_y   = 75 if floors == 2 else 105
    wall_h   = 90 if floors == 2 else 60
    roof_top = 22 if floors == 2 else 42
    floor2   = (f'<rect x="50" y="100" width="100" height="65" rx="2" '
                f'fill="#C0DD97" opacity=".4" stroke="#1D9E75" stroke-width="1"/>') if floors == 2 else ""
    label = "Double storey" if floors == 2 else "Single storey"
    return f"""
    <svg width="200" height="180" viewBox="0 0 200 180" xmlns="http://www.w3.org/2000/svg">
      <rect x="30" y="{wall_y}" width="140" height="{wall_h}" rx="4"
            fill="#EAF3DE" stroke="#1D9E75" stroke-width="1.5"/>
      {floor2}
      <polygon points="100,{roof_top} 20,{wall_y} 180,{wall_y}" fill="#1D9E75" opacity=".9"/>
      <rect x="45" y="118" width="35" height="25" rx="3" fill="#9FE1CB" stroke="#1D9E75" stroke-width="1"/>
      <rect x="120" y="118" width="35" height="25" rx="3" fill="#9FE1CB" stroke="#1D9E75" stroke-width="1"/>
      <rect x="80" y="128" width="40" height="37" rx="3" fill="#085041"/>
      <circle cx="100" cy="146" r="3" fill="#1D9E75"/>
      <text x="100" y="176" text-anchor="middle" font-size="10" fill="#3B6D11">{label}</text>
    </svg>"""


# ── CSS ───────────────────────────────────────────────────────────────────────
CSS = """
* { box-sizing: border-box; }

body, .gradio-container {
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif !important;
    background: #f0f4f8 !important;
}

.gradio-container {
    max-width: 100% !important;
    width: 100% !important;
    padding: 0 40px !important;
    margin: 0 auto !important;
}

.main  { max-width: 100% !important; }
.wrap  { max-width: 100% !important; }
footer { display: none !important; }

.page-card {
    max-width: 1100px !important;
    margin: 0 auto !important;
}

/* hide default gradio panel borders */
.gr-panel, .gr-box {
    border: none !important;
    background: transparent !important;
    padding: 0 !important;
}

/* nav bar */
#navbar {
    display: flex;
    align-items: center;
    justify-content: space-between;
    padding: 18px 0;
    border-bottom: 0.5px solid #e0e8f0;
    margin-bottom: 28px;
    max-width: 1100px;
    margin-left: auto;
    margin-right: auto;
}

/* step progress pill */
.step-pill {
    display: inline-flex;
    align-items: center;
    gap: 8px;
    background: white;
    border: 0.5px solid #e0e8f0;
    border-radius: 30px;
    padding: 6px 16px;
    margin-bottom: 20px;
}

/* green primary button */
.btn-green {
    background: linear-gradient(135deg,#1D9E75,#2EC98E) !important;
    color: white !important;
    border: none !important;
    border-radius: 12px !important;
    font-size: 15px !important;
    font-weight: 500 !important;
    padding: 14px 28px !important;
    cursor: pointer !important;
    width: 100% !important;
    box-shadow: 0 4px 14px rgba(29,158,117,.3) !important;
    transition: all .2s !important;
}
.btn-green:hover {
    transform: translateY(-1px) !important;
    box-shadow: 0 6px 20px rgba(29,158,117,.4) !important;
}

/* outline button */
.btn-outline {
    background: white !important;
    color: #444 !important;
    border: 0.5px solid #d8e0e8 !important;
    border-radius: 12px !important;
    font-size: 14px !important;
    padding: 12px 20px !important;
    cursor: pointer !important;
    width: 100% !important;
    transition: all .15s !important;
}
.btn-outline:hover {
    background: #f5f5f0 !important;
    border-color: #bbb !important;
}

/* form section cards */
.section-card {
    background: white;
    border: 0.5px solid #e0e8f0;
    border-radius: 16px;
    padding: 22px;
    margin-bottom: 14px;
    box-shadow: 0 2px 8px rgba(0,0,0,.04);
}

/* form inputs */
input[type=number], select, .gr-dropdown, .gr-number {
    border: 0.5px solid #dde4ec !important;
    border-radius: 10px !important;
    font-size: 14px !important;
    background: #fafcff !important;
    color: #1a1a1a !important;
}
input:focus, select:focus {
    border-color: #1D9E75 !important;
    outline: none !important;
    box-shadow: 0 0 0 3px rgba(29,158,117,.12) !important;
}
label {
    font-size: 12px !important;
    color: #778899 !important;
    font-weight: 500 !important;
    text-transform: uppercase !important;
    letter-spacing: .4px !important;
}

/* sliders */
input[type=range] { accent-color: #1D9E75 !important; }

/* example chips */
.chip-row { display: flex; gap: 8px; flex-wrap: wrap; }
.chip-btn {
    background: #EAF3DE !important;
    border: 0.5px solid #C0DD97 !important;
    border-radius: 20px !important;
    padding: 6px 18px !important;
    font-size: 12px !important;
    color: #27500A !important;
    cursor: pointer !important;
    white-space: nowrap !important;
    font-weight: 500 !important;
    transition: all .15s !important;
}
.chip-btn:hover {
    background: #1D9E75 !important;
    color: white !important;
    border-color: #1D9E75 !important;
}

#result-output { min-height: 200px; }
"""

# ── Navbar ────────────────────────────────────────────────────────────────────
NAVBAR = """
<div id="navbar">
  <div style="display:flex;align-items:center;gap:10px">
    <div style="width:34px;height:34px;background:linear-gradient(135deg,#1D9E75,#0F6E56);
                border-radius:10px;display:flex;align-items:center;justify-content:center">
      <svg width="18" height="18" viewBox="0 0 24 24" fill="none"
           stroke="white" stroke-width="2">
        <path d="M3 9l9-7 9 7v11a2 2 0 01-2 2H5a2 2 0 01-2-2z"/>
        <polyline points="9 22 9 12 15 12 15 22"/>
      </svg>
    </div>
    <div>
      <p style="font-size:15px;font-weight:600;color:#1a1a1a;margin:0">ISB Home Value</p>
      <p style="font-size:11px;color:#aaa;margin:0">Islamabad Real Estate</p>
    </div>
  </div>
  <div style="display:flex;align-items:center;gap:8px;
              background:white;border:0.5px solid #e0e8f0;
              border-radius:20px;padding:6px 14px">
    <div style="width:8px;height:8px;border-radius:50%;
                background:#1D9E75;box-shadow:0 0 0 2px #C6F0DF"></div>
    <span style="font-size:12px;color:#555;font-weight:500">AI Estimator · Live</span>
  </div>
</div>
"""

# ── Stats row ─────────────────────────────────────────────────────────────────
STATS_HTML = """
<div style="display:grid;grid-template-columns:repeat(3,1fr);gap:14px;margin:28px 0">
  <div style="background:white;border:0.5px solid #e0e8f0;border-radius:14px;
              padding:20px 22px;box-shadow:0 2px 8px rgba(0,0,0,.04)">
    <div style="font-size:10px;color:#aaa;text-transform:uppercase;letter-spacing:.7px">
      Listings trained on
    </div>
    <div style="font-size:32px;font-weight:600;color:#1D9E75;margin:6px 0 2px">400+</div>
    <div style="font-size:12px;color:#bbb">verified Zameen.com listings</div>
  </div>
  <div style="background:white;border:0.5px solid #e0e8f0;border-radius:14px;
              padding:20px 22px;box-shadow:0 2px 8px rgba(0,0,0,.04)">
    <div style="font-size:10px;color:#aaa;text-transform:uppercase;letter-spacing:.7px">
      ML models compared
    </div>
    <div style="font-size:32px;font-weight:600;color:#1D9E75;margin:6px 0 2px">6</div>
    <div style="font-size:12px;color:#bbb">best one auto-selected</div>
  </div>
  <div style="background:white;border:0.5px solid #e0e8f0;border-radius:14px;
              padding:20px 22px;box-shadow:0 2px 8px rgba(0,0,0,.04)">
    <div style="font-size:10px;color:#aaa;text-transform:uppercase;letter-spacing:.7px">
      City covered
    </div>
    <div style="font-size:28px;font-weight:600;color:#1D9E75;margin:6px 0 2px">
      Islamabad
    </div>
    <div style="font-size:12px;color:#bbb">Pakistan's capital city</div>
  </div>
</div>
"""

# ── Hero background with real aerial Islamabad photo ─────────────────────────
HERO_HTML = f"""
<div style="background:white;border:0.5px solid #e0e8f0;border-radius:20px;
            overflow:hidden;box-shadow:0 4px 20px rgba(0,0,0,.06);margin-bottom:4px">

  <!-- hero image strip -->
  <div style="position:relative;height:280px;overflow:hidden">
    <img
      src="https://images.unsplash.com/photo-1600585154340-be6161a56a0c?w=1200&q=80"
      alt="Beautiful Islamabad house"
      style="width:100%;height:100%;object-fit:cover;filter:brightness(.75)"
    />
    <div style="position:absolute;inset:0;
                background:linear-gradient(to right,rgba(0,0,0,.55) 0%,transparent 60%)">
    </div>
    <div style="position:absolute;top:0;left:0;right:0;bottom:0;
                display:flex;align-items:center;padding:36px 40px">
      <div>
        <div style="display:inline-flex;align-items:center;gap:6px;
                    background:rgba(255,255,255,.15);backdrop-filter:blur(6px);
                    color:white;font-size:11px;padding:5px 14px;border-radius:20px;
                    margin-bottom:18px;border:0.5px solid rgba(255,255,255,.2)">
          📍 Islamabad Real Estate · AI Powered
        </div>
        <h1 style="font-size:38px;font-weight:600;color:white;
                   line-height:1.2;margin-bottom:12px;
                   text-shadow:0 2px 12px rgba(0,0,0,.3)">
          Find your home's<br>
          <span style="color:#7FFFD4">worth in seconds.</span>
        </h1>
        <p style="font-size:15px;color:rgba(255,255,255,.85);
                  max-width:380px;line-height:1.65;margin:0">
          AI-powered estimates based on 400+ real Islamabad
          listings. Instant. Accurate. Free.
        </p>
      </div>
    </div>
    <!-- floating cards on right side of hero -->
    <div style="position:absolute;right:36px;top:50%;transform:translateY(-50%);
                display:flex;flex-direction:column;gap:10px">
      <div style="background:rgba(255,255,255,.92);backdrop-filter:blur(8px);
                  border-radius:14px;padding:14px 18px;min-width:160px;
                  box-shadow:0 4px 16px rgba(0,0,0,.12)">
        <div style="font-size:10px;color:#888;margin-bottom:4px">Average price</div>
        <div style="font-size:20px;font-weight:600;color:#1D9E75">PKR 3.5 Cr</div>
        <div style="font-size:10px;color:#aaa;margin-top:2px">Islamabad 2026</div>
      </div>
      <div style="background:rgba(255,255,255,.92);backdrop-filter:blur(8px);
                  border-radius:14px;padding:14px 18px;
                  box-shadow:0 4px 16px rgba(0,0,0,.12)">
        <div style="font-size:10px;color:#888;margin-bottom:4px">Best model</div>
        <div style="font-size:16px;font-weight:600;color:#0F6E56">CatBoost</div>
        <div style="font-size:10px;color:#aaa;margin-top:2px">R² = 0.673</div>
      </div>
    </div>
  </div>

  <!-- stats below image -->
  <div style="padding:28px 32px 32px">
    {STATS_HTML}
    <p style="font-size:12px;color:#bbb;margin:0">
      Data sourced from Zameen.com · For reference only · Not financial advice
    </p>
  </div>
</div>
"""

# ── Property photos gallery strip (shown on form page) ───────────────────────
PHOTO_STRIP = """
<div style="display:grid;grid-template-columns:repeat(3,1fr);gap:10px;
            margin-bottom:18px;border-radius:16px;overflow:hidden">
  <div style="position:relative;height:100px;overflow:hidden;border-radius:12px">
    <img src="https://images.unsplash.com/photo-1570129477492-45c003edd2be?w=400&q=80"
         style="width:100%;height:100%;object-fit:cover"/>
    <div style="position:absolute;bottom:6px;left:8px;
                background:rgba(0,0,0,.5);border-radius:8px;padding:2px 8px">
      <span style="font-size:10px;color:white">House</span>
    </div>
  </div>
  <div style="position:relative;height:100px;overflow:hidden;border-radius:12px">
    <img src="https://images.unsplash.com/photo-1545324418-cc1a3fa10c00?w=400&q=80"
         style="width:100%;height:100%;object-fit:cover"/>
    <div style="position:absolute;bottom:6px;left:8px;
                background:rgba(0,0,0,.5);border-radius:8px;padding:2px 8px">
      <span style="font-size:10px;color:white">Flat</span>
    </div>
  </div>
  <div style="position:relative;height:100px;overflow:hidden;border-radius:12px">
    <img src="https://images.unsplash.com/photo-1568605114967-8130f3a36994?w=400&q=80"
         style="width:100%;height:100%;object-fit:cover"/>
    <div style="position:absolute;bottom:6px;left:8px;
                background:rgba(0,0,0,.5);border-radius:8px;padding:2px 8px">
      <span style="font-size:10px;color:white">Farm House</span>
    </div>
  </div>
</div>
"""

# ── Gradio App ────────────────────────────────────────────────────────────────
with gr.Blocks(css=CSS, title="ISB Home Value · Islamabad House Price Predictor") as app:

    gr.HTML(NAVBAR)

    # ── PAGE 0: WELCOME ──────────────────────────────────────────────────────
    with gr.Group(visible=True) as page_welcome:
        gr.HTML(f'<div class="page-card">{HERO_HTML}</div>')
        btn_start = gr.Button(
            "✦ Get your free price estimate →",
            elem_classes=["btn-green"]
        )

    # ── PAGE 1: FORM ─────────────────────────────────────────────────────────
    with gr.Group(visible=False) as page_form:

        gr.HTML(f"""
        <div class="page-card">
          <!-- step indicator -->
          <div style="display:flex;align-items:center;gap:8px;margin-bottom:22px">
            <div style="display:flex;align-items:center;gap:6px">
              <div style="width:24px;height:24px;border-radius:50%;
                          background:#1D9E75;display:flex;align-items:center;
                          justify-content:center">
                <span style="font-size:11px;color:white;font-weight:600">1</span>
              </div>
              <span style="font-size:12px;color:#1D9E75;font-weight:500">Welcome</span>
            </div>
            <div style="flex:1;height:1px;background:#e0e8f0;margin:0 4px"></div>
            <div style="display:flex;align-items:center;gap:6px">
              <div style="width:24px;height:24px;border-radius:50%;
                          background:#1D9E75;display:flex;align-items:center;
                          justify-content:center">
                <span style="font-size:11px;color:white;font-weight:600">2</span>
              </div>
              <span style="font-size:12px;color:#1D9E75;font-weight:500">Details</span>
            </div>
            <div style="flex:1;height:1px;background:#e0e8f0;margin:0 4px"></div>
            <div style="display:flex;align-items:center;gap:6px">
              <div style="width:24px;height:24px;border-radius:50%;
                          background:#e0e8f0;display:flex;align-items:center;
                          justify-content:center">
                <span style="font-size:11px;color:#aaa;font-weight:600">3</span>
              </div>
              <span style="font-size:12px;color:#aaa">Result</span>
            </div>
          </div>

          <h2 style="font-size:24px;font-weight:600;color:#1a1a1a;margin-bottom:4px">
            Tell us about your property
          </h2>
          <p style="font-size:14px;color:#aaa;margin-bottom:22px">
            Fill in the details below for an accurate estimate
          </p>

          <!-- photo strip -->
          {PHOTO_STRIP}
        </div>
        """)

        # ── Location & type card ──────────────────────────────────
        gr.HTML("""
        <div class="page-card">
          <div style="display:flex;align-items:center;gap:8px;margin-bottom:16px">
            <div style="width:32px;height:32px;background:#EAF3DE;border-radius:10px;
                        display:flex;align-items:center;justify-content:center;font-size:16px">
              📍
            </div>
            <p style="font-size:14px;font-weight:600;color:#1a1a1a;margin:0">
              Location &amp; Type
            </p>
          </div>
        """)
        with gr.Row():
            location = gr.Dropdown(
                label="Location",
                choices=available_locations,
                value=available_locations[0],
            )
            property_type = gr.Dropdown(
                label="Property type",
                choices=["House", "Flat", "Plot", "Farm House"],
                value="House",
            )
        gr.HTML("</div>")

        # ── Area card ─────────────────────────────────────────────
        gr.HTML("""
        <div class="page-card">
          <div style="display:flex;align-items:center;gap:8px;margin-bottom:16px">
            <div style="width:32px;height:32px;background:#EAF3DE;border-radius:10px;
                        display:flex;align-items:center;justify-content:center;font-size:16px">
              📐
            </div>
            <p style="font-size:14px;font-weight:600;color:#1a1a1a;margin:0">Area Size</p>
          </div>
        """)
        with gr.Row():
            area_value = gr.Number(label="Size", value=10, minimum=1, scale=2)
            area_unit  = gr.Dropdown(
                label="Unit",
                choices=["Marla", "Kanal", "Square Feet"],
                value="Marla",
                scale=1,
            )
        gr.HTML("</div>")

        # ── Rooms card ────────────────────────────────────────────
        gr.HTML("""
        <div class="page-card">
          <div style="display:flex;align-items:center;gap:8px;margin-bottom:16px">
            <div style="width:32px;height:32px;background:#EAF3DE;border-radius:10px;
                        display:flex;align-items:center;justify-content:center;font-size:16px">
              🛏️
            </div>
            <p style="font-size:14px;font-weight:600;color:#1a1a1a;margin:0">Rooms</p>
          </div>
        """)
        bedrooms  = gr.Slider(label="Bedrooms",  minimum=1, maximum=10, step=1, value=3)
        bathrooms = gr.Slider(label="Bathrooms", minimum=1, maximum=10, step=1, value=2)
        gr.HTML("</div>")

        # ── Quick examples ────────────────────────────────────────
        gr.HTML("""
        <div class="page-card">
          <p style="font-size:11px;color:#aaa;text-transform:uppercase;
                    letter-spacing:.7px;margin-bottom:12px">
            ⚡ Quick examples — click to fill
          </p>
        """)
        with gr.Row(elem_classes=["chip-row"]):
            ex1 = gr.Button("🏠 DHA 10 Marla house",  elem_classes=["chip-btn"])
            ex2 = gr.Button("🏛️ F-7 1 Kanal house",   elem_classes=["chip-btn"])
            ex3 = gr.Button("🏢 Bahria Town flat",     elem_classes=["chip-btn"])
        gr.HTML("</div>")

        with gr.Row():
            btn_back1    = gr.Button("← Back",           elem_classes=["btn-outline"], scale=1)
            btn_estimate = gr.Button("✦ Estimate price →", elem_classes=["btn-green"],  scale=3)

    # ── PAGE 2: RESULT ────────────────────────────────────────────────────────
    with gr.Group(visible=False) as page_result:

        gr.HTML("""
        <div class="page-card">
          <div style="display:flex;align-items:center;gap:8px;margin-bottom:22px">
            <div style="display:flex;align-items:center;gap:6px">
              <div style="width:24px;height:24px;border-radius:50%;
                          background:#1D9E75;display:flex;align-items:center;
                          justify-content:center">
                <span style="font-size:11px;color:white;font-weight:600">1</span>
              </div>
              <span style="font-size:12px;color:#1D9E75;font-weight:500">Welcome</span>
            </div>
            <div style="flex:1;height:1px;background:#e0e8f0;margin:0 4px"></div>
            <div style="display:flex;align-items:center;gap:6px">
              <div style="width:24px;height:24px;border-radius:50%;
                          background:#1D9E75;display:flex;align-items:center;
                          justify-content:center">
                <span style="font-size:11px;color:white;font-weight:600">2</span>
              </div>
              <span style="font-size:12px;color:#1D9E75;font-weight:500">Details</span>
            </div>
            <div style="flex:1;height:1px;background:#e0e8f0;margin:0 4px"></div>
            <div style="display:flex;align-items:center;gap:6px">
              <div style="width:24px;height:24px;border-radius:50%;
                          background:#1D9E75;display:flex;align-items:center;
                          justify-content:center">
                <span style="font-size:11px;color:white;font-weight:600">3</span>
              </div>
              <span style="font-size:12px;color:#1D9E75;font-weight:500">Result</span>
            </div>
          </div>
          <h2 style="font-size:24px;font-weight:600;color:#1a1a1a;margin-bottom:18px">
            Your price estimate
          </h2>
        </div>
        """)

        result_html = gr.HTML(
            elem_id="result-output",
            elem_classes=["page-card"]
        )

        with gr.Row():
            btn_edit = gr.Button("← Edit details", elem_classes=["btn-outline"])
            btn_home = gr.Button("Start over",      elem_classes=["btn-outline"])

        gr.HTML("""
        <p style="font-size:12px;color:#ccc;text-align:center;
                  padding-top:16px;margin-top:4px">
          ISB Home Value · Islamabad House Price Predictor ·
          Data from Zameen.com · 400+ listings · 6 ML models evaluated
        </p>
        """)

    # ── Page transitions ──────────────────────────────────────────────────────
    def show_form():
        return gr.update(visible=False), gr.update(visible=True), gr.update(visible=False)

    def show_welcome():
        return gr.update(visible=True), gr.update(visible=False), gr.update(visible=False)

    def run_estimate(av, au, beds, baths, loc, ptype):
        html = predict_price(av, au, beds, baths, loc, ptype)
        return gr.update(visible=False), gr.update(visible=False), gr.update(visible=True), html

    btn_start.click(fn=show_form,    outputs=[page_welcome, page_form, page_result])
    btn_back1.click(fn=show_welcome, outputs=[page_welcome, page_form, page_result])
    btn_home.click( fn=show_welcome, outputs=[page_welcome, page_form, page_result])
    btn_edit.click( fn=show_form,    outputs=[page_welcome, page_form, page_result])

    btn_estimate.click(
        fn=run_estimate,
        inputs=[area_value, area_unit, bedrooms, bathrooms, location, property_type],
        outputs=[page_welcome, page_form, page_result, result_html],
    )

    # ── Example chips ─────────────────────────────────────────────────────────
    def fill_ex1():
        return 10, "Marla", 4, 3, available_locations[0], "House"

    def fill_ex2():
        idx = next((i for i, l in enumerate(available_locations) if "F-7" in l), 1)
        return 1, "Kanal", 5, 4, available_locations[idx], "House"

    def fill_ex3():
        idx = next((i for i, l in enumerate(available_locations) if "Bahria" in l), 2)
        return 5, "Marla", 2, 2, available_locations[idx], "Flat"

    for ex_btn, fill_fn in [(ex1, fill_ex1), (ex2, fill_ex2), (ex3, fill_ex3)]:
        ex_btn.click(
            fn=fill_fn,
            outputs=[area_value, area_unit, bedrooms, bathrooms, location, property_type],
        )


app.launch(share=True, inline=False, debug=False)
